# PHD from S3

In [76]:
session = boto3.Session()
session.available_profiles

['default']

In [5]:
import boto3
import pandas as pd
import io
from datetime import datetime, timedelta
import s3fs

session = boto3.Session(
    profile_name="DatafactoryAdministrator-636452271875",
    #profile_name="DatafactoryDeveloper-781690932061",
    region_name="eu-west-1"
)
s3_client = session.client("s3")


#Just to test the connection
bucket_name = "s3-dssmith-dev-costimiser"
bucket_name = "s3-dssmith-prod-costimiser"

parquet_key = "optivision/optivision_jumbo_sample_quality.parquet"
#parquet_key = "data-v1/optivision/2025/10/09/optivision_data_20251009_010740.parquet"
#parquet_key = "data-v1/optivision/2025/10/09/optivision_data_20251009_015540.parquet"
parquet_key = "data-v1/optivision/2025/10/01/optivision_data_20251001_112322.parquet"

# Get file object from S3
response = s3_client.get_object(Bucket=bucket_name, Key=parquet_key)
df = pd.read_parquet(io.BytesIO(response["Body"].read()))

print(df.head())  # Display first rows

# ===========================================================================================================================

          sample_time MBS_Current_reel_ID AB_Grade_ID  Burst_max  Burst_min  \
0 2025-10-01 11:23:22            12506543     6010120      305.0      252.0   

   Burst_samples  Burst_stdev Burst_value  Burst_var  CMT30_max  ...  \
0             22     14.55291     275.590  211.78719      188.0  ...   

   SCT_MD_max  SCT_MD_min  SCT_MD_samples SCT_MD_stdev  SCT_MD_value  \
0        4.08        3.41              22     0.199118         3.750   

   SCT_MD_var  Burst_CV%  SCT_CD_CV%  SCT_MD_CV% CMT30_CV%  
0    0.039648   5.280638    5.978842    5.309824  0.913803  

[1 rows x 31 columns]


In [16]:
from pathlib import Path
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
 
import pandas as pd
 
 
def load_parquet_data_by_date_range(
    bucket_name,
    base_prefix,
    start_date,
    end_date,
    local_cache_root="local_s3_cache",
    max_workers=8,
):
    s3_client = session.client("s3")
    local_cache_root = Path(local_cache_root)
 
    parquet_keys = []
 
    current_date = start_date
    while current_date <= end_date:
        year = current_date.strftime("%Y")
        month = current_date.strftime("%m")
        day = current_date.strftime("%d")
 
        date_prefix = f"{base_prefix}/{year}/{month}/{day}/high-frequency/"
        print(f"Checking S3 path: {date_prefix}", flush=True)
 
        paginator = s3_client.get_paginator("list_objects_v2")
        found_any = False
 
        for page in paginator.paginate(Bucket=bucket_name, Prefix=date_prefix):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                if key.endswith(".parquet"):
                    parquet_keys.append(key)
                    found_any = True
 
        if not found_any:
            print(f"No parquet data found for {current_date.strftime('%Y-%m-%d')}.", flush=True)
 
        current_date += timedelta(days=1)
 
    if not parquet_keys:
        print("No parquet files found in the given date range.")
        return pd.DataFrame()
 
    def load_one_file(file_key):
        local_path = local_cache_root / file_key
        local_path.parent.mkdir(parents=True, exist_ok=True)
 
        try:
            if local_path.exists():
                print(f"Reading local: {local_path}", flush=True)
            else:
                print(f"Downloading: {file_key}", flush=True)
                s3_client.download_file(bucket_name, file_key, str(local_path))
 
            df = pd.read_parquet(local_path)
            df["source_file"] = file_key
            return df
 
        except Exception as e:
            print(f"Error loading {file_key}: {e}", flush=True)
            return None
 
    data_frames = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(load_one_file, key) for key in parquet_keys]
 
        for future in as_completed(futures):
            df = future.result()
            if df is not None:
                data_frames.append(df)
 
    if not data_frames:
        print("Files were found, but none could be loaded.")
        return pd.DataFrame()
 
    return pd.concat(data_frames, ignore_index=True)
 
 
bucket_name = "s3-dssmith-prod-costimiser"
base_prefix = "data-v1/raw-fetched-data"
 
start_date = datetime(2026, 3, 1)
end_date = datetime(2026, 3, 3)
 
df = load_parquet_data_by_date_range(
    bucket_name=bucket_name,
    base_prefix=base_prefix,
    start_date=start_date,
    end_date=end_date,
    local_cache_root="local_s3_cache",
    max_workers=8,
)
 
print(df.shape)
print(df.head())

Checking S3 path: data-v1/raw-fetched-data/2026/03/01/high-frequency/
Checking S3 path: data-v1/raw-fetched-data/2026/03/02/high-frequency/
Checking S3 path: data-v1/raw-fetched-data/2026/03/03/high-frequency/
Reading local: local_s3_cache\data-v1\raw-fetched-data\2026\03\01\high-frequency\phd_data_20260301_000000.parquet
Reading local: local_s3_cache\data-v1\raw-fetched-data\2026\03\01\high-frequency\phd_data_20260301_000100.parquet
Reading local: local_s3_cache\data-v1\raw-fetched-data\2026\03\01\high-frequency\phd_data_20260301_000200.parquet
Reading local: local_s3_cache\data-v1\raw-fetched-data\2026\03\01\high-frequency\phd_data_20260301_000300.parquet
Reading local: local_s3_cache\data-v1\raw-fetched-data\2026\03\01\high-frequency\phd_data_20260301_000400.parquet
Reading local: local_s3_cache\data-v1\raw-fetched-data\2026\03\01\high-frequency\phd_data_20260301_000500.parquet
Reading local: local_s3_cache\data-v1\raw-fetched-data\2026\03\01\high-frequency\phd_data_20260301_000600.

In [ ]:
df.to_parquet(f"data/Costimier_PHD_{pd.to_datetime(start_date).strftime("%Y-%m-%d")}.parquet", index=False)

# PHD from WS

In [ ]:
import requests
from tqdm.notebook import tqdm
from datetime import date, timedelta, datetime
import pandas as pd

def daterange(start_date, end_date, jump="day"):
    from datetime import date, timedelta
    import pandas as pd

    d = start_date
    while d <= end_date:
        yield d
        if jump=="week":
            d += timedelta(days=7)
        elif jump=="month":
            d += pd.DateOffset(months=1)
        elif jump=="hour":
            d += pd.DateOffset(hours=1)
        else:
            d += timedelta(days=1)

# tag_list = ["R2NAE20CF905"]

df_tags = pd.read_csv("data/tags-v3.csv", sep=';')  # Adjusted to match the CSV format used in the original code
df_tags['TagNumber'] = df_tags['TagNumber'].astype(float)
tag_list = df_tags['Tags'][df_tags['TagNumber'].notna()]
tag_list = tag_list.to_list()


def convert_to_dataframe(data: pd.DataFrame, tag_list:list) -> pd.DataFrame:
    dfs = []
    i=0
    for d in data:
        tag_name = d['TagName']
        timestamps = d['TimeStamp']
        values = d['Value']
                   
        if i==0:
            df_data = {
                'TimeStamp': timestamps,
                f'{tag_name}': values
            }
        else:
            df_data = {
                f'{tag_name}': values
            }

        df = pd.DataFrame(df_data)
        dfs.append(df,)
        i=i+1

    result_df = pd.concat(dfs, axis=1)
    result_df = result_df[['TimeStamp'] + tag_list]

    return result_df

for d in tqdm(daterange(datetime(2026, 3, 2, 0), datetime.now(), jump="hour")):
    print(d)
    start_time = d
    end_time = d + timedelta(hours=1)
    params = {
        "SampleInterval": 60000,        
        "ResampleMethod": "Around",      
        "MaxRows": 10000,
        "TimeFormat": 1,
        "TagName": tag_list,
        "StartTime": start_time.strftime("%d-%b-%Y %H:%M:%S.%f")[:-3],
        "EndTime": end_time.strftime("%d-%b-%Y %H:%M:%S.%f")[:-3],
        "OutputTimeFormat": 1
        }
    
    response = requests.post('https://dss-de-ab-phd7.dss.dssmithgroup.local:3152/GetData', json=params, verify=False, timeout=240).json()
    df_phd = convert_to_dataframe(response, tag_list)
    df_phd["TimeStamp"]=pd.to_datetime(df_phd["TimeStamp"], format="%d-%b-%Y %H:%M:%S")
    #df_phd=df_phd.set_index("TimeStamp")
    df_phd.to_parquet(f"data-v2/raw-fetched-data/phd_data_{start_time.strftime("%Y%m%d_0%H000")}.parquet", index=False)

In [2]:
import os
import pyarrow.parquet as pq
import pyarrow as pa
dataset = pq.ParquetDataset("data-v2/raw-fetched-data/")
table = dataset.read()

new_fields = []
for field in table.schema:
    if pa.types.is_float64(field.type):
        new_fields.append(pa.field(field.name, pa.float32()))
    
    elif pa.types.is_int64(field.type):
        new_fields.append(pa.field(field.name, pa.int32()))
    else:
        new_fields.append(field)

new_schema = pa.schema(new_fields)
table = table.cast(new_schema)
df_phd = table.to_pandas()

df_phd.to_parquet(f"phd/phd.parquet", index=False)

In [ ]:
import os
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
 
SRC_DIR = "data-v2/raw-fetched-data/"
OUT_FILE = "phd/phd_all_casted.parquet"
 
Path(OUT_FILE).parent.mkdir(parents=True, exist_ok=True)
 
def build_cast_schema(schema: pa.Schema) -> pa.Schema:
    fields = []
    for f in schema:
        t = f.type
        if pa.types.is_float64(t):
            fields.append(pa.field(f.name, pa.float32(), f.nullable, f.metadata))
        elif pa.types.is_int64(t):
            fields.append(pa.field(f.name, pa.int32(), f.nullable, f.metadata))
        else:
            fields.append(f)
    return pa.schema(fields)
 
# Collect input files
in_files = []
for root, _, files in os.walk(SRC_DIR):
    for fn in files:
        if fn.endswith(".parquet"):
            in_files.append(os.path.join(root, fn))
 
if not in_files:
    raise FileNotFoundError("No parquet files found")
 
print(f"Found {len(in_files)} parquet files")
 
writer = None
rows_written = 0
 
try:
    for i, in_path in enumerate(sorted(in_files), 1):
        pf = pq.ParquetFile(in_path)
 
        # Build schema from the FIRST file only
        if writer is None:
            target_schema = build_cast_schema(pf.schema_arrow)
            writer = pq.ParquetWriter(
                OUT_FILE,
                target_schema,
                compression="snappy",
                use_dictionary=False,
                write_statistics=False,
            )
 
        print(f"[{i}/{len(in_files)}] Processing {os.path.basename(in_path)}")
 
        for batch in pf.iter_batches(batch_size=20_000, use_threads=False):
            batch = batch.cast(target_schema, safe=False)
            table = pa.Table.from_batches([batch], schema=target_schema)
            writer.write_table(table, row_group_size=batch.num_rows)
            rows_written += batch.num_rows
 
finally:
    if writer is not None:
        writer.close()
 
print(f"✅ Done. Rows written: {rows_written:,}")
print(f"Output file: {OUT_FILE}")

## Query all tags

In [42]:
import os
import requests
from tqdm.notebook import tqdm
from datetime import datetime, timedelta
import pandas as pd
 
def daterange(start_date, end_date, jump="day"):
    d = start_date
    while d <= end_date:
        yield d
        if jump == "week":
            d += timedelta(days=7)
        elif jump == "month":
            d += pd.DateOffset(months=1)
        elif jump == "hour":
            d += pd.DateOffset(hours=1)
        else:
            d += timedelta(days=1)
 
df_tags = pd.read_csv("data/tags-v3.csv", sep=';')
df_tags["TagNumber"] = pd.to_numeric(df_tags["TagNumber"], errors="coerce")
 
tag_list = (
    df_tags.loc[df_tags["TagNumber"].notna(), "Tags"]
    .astype(str)
    .tolist()
)
tag_list = ['QO-REEL-ID-CUR-CALC']
 
def convert_to_dataframe(data, tag_list):
    # data expected: list of dicts with TagName, TimeStamp, Value
    if not data:
        return pd.DataFrame(columns=["TimeStamp"] + tag_list)
 
    dfs = []
    for i, d in enumerate(data):
        tag_name = d.get("TagName")
        timestamps = d.get("TimeStamp", [])
        values = d.get("Value", [])
 
        if i == 0:
            df = pd.DataFrame({"TimeStamp": timestamps, tag_name: values})
        else:
            df = pd.DataFrame({tag_name: values})
        dfs.append(df)
 
    result_df = pd.concat(dfs, axis=1)
 
    # Ensure we have all requested tags (some might be missing in response)
    for t in tag_list:
        if t not in result_df.columns:
            result_df[t] = pd.NA
 
    return result_df[["TimeStamp"] + tag_list]
 
# -----------------------------
# Daily parquet writer
# -----------------------------
OUT_DIR = "data-v4/raw-fetched-data"
os.makedirs(OUT_DIR, exist_ok=True)
 
def write_daily_parquet(day_df: pd.DataFrame, day: datetime):
    if day_df is None or day_df.empty:
        return
    day_str = day.strftime("%Y%m%d")
    out_path = os.path.join(OUT_DIR, f"phd_data_{day_str}.parquet")
 
    # Clean up / keep one row per timestamp
    day_df = day_df.sort_values("TimeStamp").drop_duplicates(subset=["TimeStamp"], keep="last")
 
    # Optional: enforce dtypes (timestamps already datetime64)
    day_df.to_parquet(out_path, index=False)
 
# Choose your start and end
start_dt = datetime(2025, 4, 1, 0)
end_dt = datetime(2026, 2, 18, 0)
 
current_day = None
daily_parts = []
 
session = requests.Session()
 
for d in tqdm(daterange(start_dt, end_dt, jump="hour")):
    start_time = d
    end_time = d + timedelta(hours=1)
 
    # Track day boundaries based on start_time date
    day_key = start_time.date()
    if current_day is None:
        current_day = day_key
 
    # If day changed, flush previous day to parquet
    if day_key != current_day:
        day_df = pd.concat(daily_parts, ignore_index=True) if daily_parts else pd.DataFrame()
        write_daily_parquet(day_df, datetime.combine(current_day, datetime.min.time()))
        daily_parts = []
        current_day = day_key
 
    params = {
        "SampleInterval": 60000,
        "ResampleMethod": "Around",
        "MaxRows": 10000,
        "TimeFormat": 1,
        "TagName": tag_list,
        "StartTime": start_time.strftime("%d-%b-%Y %H:%M:%S.%f")[:-3],
        "EndTime": end_time.strftime("%d-%b-%Y %H:%M:%S.%f")[:-3],
        "OutputTimeFormat": 1
    }
 
    try:
        r = session.post(
            "https://dss-de-ab-phd7.dss.dssmithgroup.local:3152/GetData",
            json=params,
            verify=False,
            timeout=240
        )
        r.raise_for_status()
        response = r.json()
    except Exception as e:
        # If the API flakes, keep going and just skip this hour
        print(f"Failed {start_time} -> {end_time}: {e}")
        continue
 
    df_phd = convert_to_dataframe(response, tag_list)
 
    # Parse timestamps robustly; your format example is "01-Dec-2025 13:00:00"
    df_phd["TimeStamp"] = pd.to_datetime(df_phd["TimeStamp"], errors="coerce", format="%d-%b-%Y %H:%M:%S")
 
    # Drop rows where timestamp couldn't be parsed
    df_phd = df_phd.dropna(subset=["TimeStamp"])
 
    daily_parts.append(df_phd)
 
# Flush the last day
if current_day is not None:
    day_df = pd.concat(daily_parts, ignore_index=True) if daily_parts else pd.DataFrame()
    write_daily_parquet(day_df, datetime.combine(current_day, datetime.min.time()))

0it [00:00, ?it/s]

c:\workspace\costimiser\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'dss-de-ab-phd7.dss.dssmithgroup.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\workspace\costimiser\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'dss-de-ab-phd7.dss.dssmithgroup.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\workspace\costimiser\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'dss-de-ab-phd7.dss.dssmithgroup.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-war

In [ ]:
import os
import pandas as pd
from pathlib import Path
 
# -----------------------------
# Paths
# -----------------------------
DIR_V3 = Path("data-v3/raw-fetched-data")
DIR_V4 = Path("data-v4/raw-fetched-data")
 
# Optional: write to new folder instead of overwriting
OUT_DIR = Path("data-v3-merged/raw-fetched-data")
OUT_DIR.mkdir(parents=True, exist_ok=True)
 
# -----------------------------
# Get common parquet files
# -----------------------------
files_v3 = {f.name for f in DIR_V3.glob("phd_data_*.parquet")}
files_v4 = {f.name for f in DIR_V4.glob("phd_data_*.parquet")}
 
common_files = sorted(files_v3.intersection(files_v4))
 
print(f"Found {len(common_files)} common files")
 
# -----------------------------
# Merge loop
# -----------------------------
for fname in common_files:
    print(f"Merging {fname}")
 
    path_v3 = DIR_V3 / fname
    path_v4 = DIR_V4 / fname
 
    df_v3 = pd.read_parquet(path_v3)
    df_v4 = pd.read_parquet(path_v4)
 
    # Ensure TimeStamp is datetime
    df_v3["TimeStamp"] = pd.to_datetime(df_v3["TimeStamp"])
    df_v4["TimeStamp"] = pd.to_datetime(df_v4["TimeStamp"])
 
    # Identify new columns in v4
    new_cols = [c for c in df_v4.columns if c not in df_v3.columns]
 
    if not new_cols:
        print(f"  No new columns in {fname}")
        merged_df = df_v3
    else:
        print(f"  Adding columns: {new_cols}")
 
        merged_df = df_v3.merge(
            df_v4[["TimeStamp"] + new_cols],
            on="TimeStamp",
            how="left"
        )
 
    # Optional: sort & deduplicate
    merged_df = (
        merged_df
        .sort_values("TimeStamp")
        .drop_duplicates(subset=["TimeStamp"], keep="last")
    )
 
    # Write result
    merged_df.to_parquet(OUT_DIR / fname, index=False)
 
print("Done.")

# Optivision from S3

In [86]:
import boto3
import pandas as pd
import io
from datetime import datetime, timedelta
import s3fs

session = boto3.Session(
    profile_name="DatafactoryAdministrator-636452271875",
    #profile_name="DatafactoryDeveloper-781690932061",
    region_name="eu-west-1"
)
s3_client = session.client("s3")


#Just to test the connection
#bucket_name = "s3-dssmith-dev-costimiser"
bucket_name = "s3-dssmith-prod-costimiser"

parquet_key = "optivision/optivision_jumbo_sample_quality.parquet"
#parquet_key = "data-v1/optivision/2025/10/09/optivision_data_20251009_010740.parquet"
#parquet_key = "data-v1/optivision/2025/10/09/optivision_data_20251009_015540.parquet"
parquet_key = "data-v1/optivision/2025/10/01/optivision_data_20251001_112322.parquet"

# Get file object from S3
response = s3_client.get_object(Bucket=bucket_name, Key=parquet_key)
df = pd.read_parquet(io.BytesIO(response["Body"].read()))

print(df.head())  # Display first rows



          sample_time MBS_Current_reel_ID AB_Grade_ID  Burst_max  Burst_min  \
0 2025-10-01 11:23:22            12506543     6010120      305.0      252.0   

   Burst_samples  Burst_stdev Burst_value  Burst_var  CMT30_max  ...  \
0             22     14.55291     275.590  211.78719      188.0  ...   

   SCT_MD_max  SCT_MD_min  SCT_MD_samples SCT_MD_stdev  SCT_MD_value  \
0        4.08        3.41              22     0.199118         3.750   

   SCT_MD_var  Burst_CV%  SCT_CD_CV%  SCT_MD_CV% CMT30_CV%  
0    0.039648   5.280638    5.978842    5.309824  0.913803  

[1 rows x 31 columns]


In [ ]:
import os
from pathlib import Path
from typing import List, Iterable
from urllib.parse import urlparse
 
import pandas as pd
import boto3
 
 
def list_prefixes(s3, bucket: str, prefix: str = "", max_keys: int = 1000) -> List[str]:
    """
    List immediate 'subfolders' under prefix using Delimiter='/'.
    Returns a list of S3 keys that end with '/'.
    """
    paginator = s3.get_paginator("list_objects_v2")
    result: List[str] = []
    for page in paginator.paginate(
        Bucket=bucket,
        Prefix=prefix,
        Delimiter="/",
        PaginationConfig={"PageSize": max_keys},
    ):
        for cp in page.get("CommonPrefixes", []):
            result.append(cp["Prefix"])
    return result
 
 
def walk(s3, bucket: str, prefix: str = "") -> Iterable[str]:
    """
    Recursively yield all object keys under prefix.
    """
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if not key.endswith("/"):
                yield key
 
 
def find_file(s3, bucket: str, prefix: str = "", extension: str = "parquet") -> List[str]:
    """
    Find all object URIs with the given extension under prefix (recursive).
    Returns s3:// URIs.
    """
    keys = [k for k in walk(s3, bucket, prefix) if k.lower().endswith(f".{extension.lower()}")]
    return [f"s3://{bucket}/{k}" for k in keys]
 
 
def show_files(s3, bucket, base="", extension="parquet"):
    parquet_paths = find_file(s3, bucket, base, extension)
    print(f"\nData files found recursively under '{base}': {len(parquet_paths)}")
    for p in parquet_paths[:5]:
        print("  -", p)
    if not parquet_paths:
        print("No data files detected under this prefix.")
    return parquet_paths
 
 
def s3_uri_to_bucket_key(s3_uri: str):
    """
    Split s3://bucket/key into bucket, key
    """
    parsed = urlparse(s3_uri)
    bucket = parsed.netloc
    key = parsed.path.lstrip("/")
    return bucket, key
 
 
def s3_uri_to_local_path(s3_uri: str, local_cache_dir: str) -> Path:
    """
    Preserve the S3 key structure under a local cache directory.
    Example:
      s3://my-bucket/data-v1/optivision/2025/file.parquet
      -> ./local_s3_cache/my-bucket/data-v1/optivision/2025/file.parquet
    """
    bucket, key = s3_uri_to_bucket_key(s3_uri)
    return Path(local_cache_dir) / bucket / key
 
 
def ensure_local_file(s3_client, s3_uri: str, local_cache_dir: str) -> str:
    """
    If file exists locally, reuse it.
    Otherwise download it from S3 and return the local path.
    """
    bucket, key = s3_uri_to_bucket_key(s3_uri)
    local_path = s3_uri_to_local_path(s3_uri, local_cache_dir)
 
    local_path.parent.mkdir(parents=True, exist_ok=True)
 
    if local_path.exists():
        print(f"Using local file: {local_path}")
    else:
        print(f"Downloading {s3_uri} -> {local_path}")
        s3_client.download_file(bucket, key, str(local_path))
 
    return str(local_path)
 
 
# -----------------------------
# Main
# -----------------------------
prefixes = [
    "data-v1/optivision/2025/",
    "data-v1/optivision/2026/",
]
 
local_cache_dir = "./local_s3_cache"
 
file_list = []
for base in prefixes:
    file_list.extend(
        show_files(
            s3_client,
            bucket_name,
            base=base,
            extension="parquet",
        )
    )
 
# Download only if missing locally
local_files = [ensure_local_file(s3_client, s3_uri, local_cache_dir) for s3_uri in file_list]
 
# Read local parquet files
dfs = [pd.read_parquet(p) for p in local_files]
optvsn_df = pd.concat(dfs, ignore_index=True, sort=False)
 
exclude_cols = {"sample_time", "MBS_Current_reel_ID", "AB_Grade_ID"}
colnum = [c for c in optvsn_df.columns if c not in exclude_cols]
optvsn_df[colnum] = optvsn_df[colnum].apply(pd.to_numeric, errors="coerce")
optvsn_df.to_parquet(f"phd/optvsn_df.parquet")

# Optivision from SQL Server

In [ ]:
import json
import os
import pyarrow.parquet as pq
from typing import List, Optional, Iterable, Dict, Tuple
import pandas as pd
import pyarrow.dataset as ds
import s3fs
import boto3, botocore.session, s3fs

session = boto3.Session(
    #profile_name="DatafactoryAdministrator-636452271875",
    profile_name="DatafactoryDeveloper-781690932061",
    region_name="eu-west-1"
)
s3_client = session.client("s3")


bucket_name = 's3-dssmith-dev-costimiser'
bucket_region = 'eu-west-1'


ssm = session.client('ssm', region_name=bucket_region)
parameter_name = "/service/costimiser-optivision-fetch/secret"
try:
    response = ssm.get_parameter(
        Name=parameter_name,
        WithDecryption=True
    )
    parameter_value = response['Parameter']['Value']
    secret_data = json.loads(parameter_value)

    optivision_database = secret_data.get('OptivisionDatabase')
    optivision_fqdn = secret_data.get('OptivisionFqdn')
    optivision_user = secret_data.get('OptivisionUser')
    optivision_password = secret_data.get('OptivisionPassword')
except Exception as e:
    print(f"Can't fetch secrets: {str(e)}")
    

server = optivision_fqdn
database = optivision_database
username = optivision_user
password = optivision_password

In [ ]:
print(server)
print(database)
print(username)
print(password)

dss-de-ab-db1.dss.dssmithgroup.local
ABPROD
Cost_views
CoSt3$5ImiZer!


In [ ]:
import pyodbc

conn_str = f"DRIVER={{SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}"

try:
    # Set query timeout to 20 seconds
    conn = pyodbc.connect(conn_str, timeout=20)
    cursor = conn.cursor()

    print("Test reading data from view dss_jumbo_Q_stats\n")
    # Define view name
    view_name = "dss_jumbo_Q_stats"

    try:
        # Run the query to read the view
        # query = f"SELECT TOP 1000 * FROM {view_name} ORDER BY sample_time DESC;"
        query = f"""
            SELECT * FROM {view_name}
            WHERE sample_time >= '2025-1-1'
            ORDER BY sample_time DESC, spec_name;
        """
        cursor.execute(query)

        # Retrieve columns dynamically
        columns = [col[0] for col in cursor.description]  # Get column names
        rows = cursor.fetchall()  # Fetch all rows

        # Debugging: Ensure fetched data is correctly structured
        print(f"Number of rows fetched: {len(rows)}")
        print(f"Columns: {columns}")
        print(f"Row type: {type(rows[0])} with length: {len(rows[0])}" if rows else "No data returned.")

        # Convert pyodbc.Row objects into standard Python tuples
        formatted_rows = [tuple(row) for row in rows]

        # Create DataFrame
        df = pd.DataFrame(formatted_rows, columns=columns)
        print(df.head())

        # # the data will be saved as a parquet file
        # filename = "optivision_jumbo_sample_quality"
        # status_prefix = "data-v1/optivision"
        # save_to_s3(df, filename, status_prefix)

        # # Ensure column names are clean
        # df.columns = df.columns.str.strip()
        # # This checks every value in the dataframe and only applies .strip() to strings.
        # df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
        # # rename columns to correspond to PHD information
        # df = df.rename(columns={'jumbo_id': 'MBS_Current_reel_ID', 'grade_spec': 'AB_Grade_ID'})
        # # Keep only instances for SCT CD, Burst, SCT MD and CMT30
        # df = df[df['spec_name'].isin(['SCT CD', 'Berstwiderstand', 'SCT MD', 'CMT', 'Feu AL400'])].reset_index(drop=True)
        # # Melt the dataframe to bring all metrics under a single column
        # df_melted = df.melt(id_vars=['sample_time', 'MBS_Current_reel_ID', 'AB_Grade_ID', 'spec_name'], value_vars=['value', 'max', 'min', 'stdev', 'var', 'samples'], var_name='metric', value_name='measurement')
        # # Pivot to create the final dataframe
        # df_pivot = df_melted.pivot(index=['sample_time', 'MBS_Current_reel_ID', 'AB_Grade_ID', 'metric'], columns='spec_name', values='measurement').reset_index()
        # target_dict = {'Berstwiderstand': 'Burst', 'SCT CD': 'SCT_CD', 'SCT MD': 'SCT_MD', 'CMT': 'CMT30', 'Feu AL400': 'Moisture_Test_Line'}
        # df_pivot = df_pivot.rename(columns=target_dict)
        # # Reshape the dataframe to have separate columns for each metric
        # df_final = df_pivot.pivot(index=['sample_time', 'MBS_Current_reel_ID', 'AB_Grade_ID'], columns='metric').reset_index()
        # # Flatten the multi-level column names
        # df_final.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col for col in df_final.columns]
        # # Calculate Coefficient of Variation (CV) for each spec_name
        # from decimal import Decimal
        # for spec in target_dict.values():
        #     if f"{spec}_stdev" in df_final.columns and f"{spec}_value" in df_final.columns:
        #         df_final[f"{spec}_CV%"] = (df_final[f"{spec}_stdev"].astype(float)/df_final[f"{spec}_value"].replace(0, pd.NA).astype(float))*100

        

    except Exception as e:
        print(f"Database query error: {e}")

    cursor.close()
    conn.close()

except pyodbc.Error as e:
    print(f"Error connecting to the database: {e}")
 

Test reading data from view dss_jumbo_Q_stats

Number of rows fetched: 10647
Columns: ['Mill', 'PM', 'jumbo_id', 'grade_spec', 'sample_time', 'spec_name', 'standard_spec', 'value', 'unit_of_measure', 'min', 'max', 'stdev', 'var', 'samples', 'profile']
Row type: <class 'pyodbc.Row'> with length: 15
  Mill  PM  jumbo_id grade_spec         sample_time        spec_name  \
0   AB  01  12507811    6010120 2025-11-19 11:32:35  Berstwiderstand   
1   AB  01  12507811    6010120 2025-11-19 11:32:35            Dicke   
2   AB  01  12507811    6010120 2025-11-19 11:32:35        Flg AL400   
3   AB  01  12507811    6010120 2025-11-19 11:32:35           Gurley   
4   AB  01  12507811    6010120 2025-11-19 11:32:35           SCT CD   

  standard_spec    value unit_of_measure    min     max      stdev  \
0             6  264.390                  242.0  280.00  11.071857   
1            48  172.300                  168.0  181.00   2.741545   
2            54  118.490                  117.1  120.70   

In [ ]:
df.sample_time.max()

Timestamp('2025-11-19 11:32:35')

In [ ]:
df.sample_time.min()

Timestamp('2025-10-09 22:12:05')

In [ ]:
df.spec_name.unique()

array(['Berstwiderstand', 'Dicke', 'Flg AL400', 'Gurley', 'SCT CD',
       'SCT MD', 'TSI CD', 'TSI MD', 'TSO-Winkel', 'Zugsteifigk cd',
       'Zugsteifigk md', 'CMT'], dtype=object)

In [ ]:
df.to_parquet(f"phd/lab.parquet", index=False)

# Generate turnup

In [87]:


import boto3
import pandas as pd
import io
from datetime import datetime, timedelta
import s3fs

session = boto3.Session(
    #profile_name="DatafactoryAdministrator-636452271875",
    profile_name="DatafactoryDeveloper-781690932061",
    region_name="eu-west-1"
)
s3_client = session.client("s3")


#Just to test the connection
bucket_name = "s3-dssmith-dev-costimiser"
#bucket_name = "s3-dssmith-prod-costimiser"

parquet_key = "optivision/optivision_jumbo_sample_quality.parquet"
#parquet_key = "data-v1/optivision/2025/10/09/optivision_data_20251009_010740.parquet"
#parquet_key = "data-v1/optivision/2025/10/09/optivision_data_20251009_015540.parquet"
parquet_key = "data-v1/optivision/2025/10/01/optivision_data_20251001_112322.parquet"

# Get file object from S3
response = s3_client.get_object(Bucket=bucket_name, Key=parquet_key)
df = pd.read_parquet(io.BytesIO(response["Body"].read()))

print(df.head())  # Display first rows

          sample_time MBS_Current_reel_ID AB_Grade_ID  Burst_max  Burst_min  \
0 2025-10-01 11:23:22            12506543     6010120      305.0      252.0   

   Burst_samples  Burst_stdev Burst_value  Burst_var  CMT30_max  ...  \
0             22     14.55291     275.590  211.78719      188.0  ...   

   SCT_MD_max  SCT_MD_min  SCT_MD_samples SCT_MD_stdev  SCT_MD_value  \
0        4.08        3.41              22     0.199118         3.750   

   SCT_MD_var  Burst_CV%  SCT_CD_CV%  SCT_MD_CV% CMT30_CV%  
0    0.039648   5.280638    5.978842    5.309824  0.913803  

[1 rows x 31 columns]


## Legacy

In [1]:
import pandas as pd

time_ref="2025-12-01"
df_cost_ts = pd.read_parquet(f"data/Costimier_PHD_{time_ref}.parquet",engine="fastparquet")
 
df_cost_ts["TimeStamp"]=pd.to_datetime(df_cost_ts["TimeStamp"],format='ISO8601')
df_cost_ts=df_cost_ts.sort_values(by='TimeStamp')
print(f"Time Period: {df_cost_ts['TimeStamp'].min()}, {df_cost_ts['TimeStamp'].max()}")

Time Period: 2025-12-01 00:00:00, 2025-12-31 16:39:00


In [2]:
df_tags = pd.read_csv("data/tags-v3.csv", sep=';')  # Adjusted to match the CSV format used in the original code
df_tags['TagNumber'] = df_tags['TagNumber'].astype(float)
tag_list = df_tags['Tags'][df_tags['TagNumber'].notna()]
tag_list = tag_list.to_list()

df_gloss = pd.read_csv("data/data_glossary_3.csv", sep=';')

In [3]:
import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np


from src.ingestion_pipeline import ingestion_pipeline_API_alternative
from src.alignment_pipeline import data_alignment, assign_missing_values_to_grades_without_cmt30, drop_grades_equal_to_zero, drop_paperbreaks,create_turnup_and_continuous_data
from src.process_data_treatment  import remove_empty_columns, drop_rows_with_missing_values, forward_fill_by_group, replace_placeholder_values, forward_fill_by_group, fill_na_with_median
from utility import add_germany_temperature_column

In [7]:
control_vars = [
    "Current_basis_weight", # Scanner
    "Speed", # Scanner
    "Current_reel_width",
    "Current_reel_moisture_average(reel)", # Scanner
    "Actual_moisture", # (Moisture before SpeedSizer) Scanner
    "Sizing_Agent__g/T_", # Size Press
    "Draw_PD5-SS",  # Size Press
    "Defoamer_mass_flow__g/T_", # Size Press
    "Dry_Strength_Agent_mass_flow__kg/T_", # Size Press
    "SpeedSizer_Linepressure_DS", # Size Press
    "SpeedSizer_Linepressure_FS", # Size Press
    "Consistency_starch_main_line", # Size Press
    "Draw_PS-PD1", #Pre-dryer
    "Draw_PD2-PD3", #Pre-dryer
    "Draw_PD4-PD5", #Pre-dryer
    "Draw_PD3-PD4", #Pre-dryer
    "Draw_PD1-PD2", #Pre-dryer
    "Moisture_out_of_PreDryer", #Pre-dryer
    "Draw_WS-PS", # Press Section
    "PickUp_Tension", # Press Section
    "Vacuum_presszone_of_suction-press_roll", # Press Section
    "Vacuum_uhle-box_Pick-Up", # Press Section
    "Vacuum_uhle-box_bottom_felt", # Press Section
    "Linepressure_1st_press_FS__bar_", # Press Section
    "Linepressure_2nd_press_FS__bar_", # Press Section
    "Linepressure_1st_press_DS__bar_", # Press Section
    "Linepressure_2nd_press_DS__bar_", # Press Section
    "Linepressure_shoe_press__bar_", # Press Section
    "Dewatering_top_wire_suction_box_zone_2", # Forming Wire
    "Vacuum_suction_box_9", "Vacuum_wet_suction_box", # Forming Wire
    "Vacuum_sheet_seperator_box", # Forming Wire
    "Vacuum_suction_box_10", # Forming Wire
    "Vacuum_suction_box_11", # Forming Wire
    "Vacuum_wire_suction_box_1", # Forming Wire
    "Vacuum_wire_suction_box_2", # Forming Wire
    "Consistency_white_water", # Forming Wire
    "White_water_temperature", # Forming Wire
    "Conductivity_white_water_B46", # Forming Wire
    "Top_wire_tenstion", # Forming Wire
    "pH_measurement_white_water_B41", # Forming Wire
    "Jet/wire_ratio", # Headbox
    "Lip_settings", # Headbox
    "Retention_Aid_mass_flow__g/T_", # Approach Flow
    "Bentonite_1_mass_flow__g/T_", # Approach Flow
    "Bentonite_2_mass_flow__g/T_", # Approach Flow
    "Thick_Stock_Consistency__%_", # Approach Flow
]
 
response_vars=[
    "Electricity__kWh/T_",
    "Production_Rate__T/h_",
    "Steam_flow_to_PreDryers",
    "Steam_flow_to_AfterDryers",
    'Steam__kWh/T_'
] 

In [8]:
df_process_ts  = ingestion_pipeline_API_alternative(df_cost_ts, df_tags, df_gloss)
df_process_ts['Wedge_Time'] = pd.to_datetime(df_process_ts['Wedge_Time'])
 
quality_list = ['MBS_SCT_MD', 'MBS_SCT_CD', 'MBS_Burst', 'MBS_CMT30']
data_aligned_continuous = data_alignment(df_process_ts, quality_list)
 
"""
ASSIGN MISSING VALUES TO QUALITY VALUES FOR GRADES THAT DON'T HAVE CMT30
"""
# Defining grades that dont have CMT30
grade_list = [3600075, 3600085,6010075,6010080,6010085,6010090,6010095,3200100,3200115,3200125,3300085,
            3300090,3300095,3300100,3300105,3300110,3300115,3300120,3300125,3300130,3300135]
data_aligned_continuous = assign_missing_values_to_grades_without_cmt30(data_aligned_continuous, grade_list)
 
# Drop zero grades
data_aligned_continuous = drop_grades_equal_to_zero(data_aligned_continuous)
 
# Drop rows with paperbreaks
data_aligned_continuous = drop_paperbreaks(data_aligned_continuous)
 
# Step 1: Remove columns that are completely missing
df_treated = remove_empty_columns(data_aligned_continuous)
 
# Step 2: Drop rows where specific columns have NaN values
columns_to_verify_missing = [
    "Starch__€/T_",
    "Dry_Strength_Agent__€/T_",
    "Fibre_cost__€/T_",
    "Combined_cost__€/T_",
    "Starch_mass_flow__kg/T_",
    "Dry_Strength_Agent_mass_flow__kg/T_"]
df_treated = drop_rows_with_missing_values(df_treated, columns_to_verify_missing)
 
# Step 3: Forward-fill within each MBS_Current_reel_ID group
df_treated = forward_fill_by_group(df_treated, 'MBS_Current_reel_ID')
 
# Step 4: Replace placeholder values (999999) with NaN
df_treated = replace_placeholder_values(df_treated)
 
# Step 5: Forward-fill again within each MBS_Current_reel_ID group (for NaNs introduced by replacing 999999)
df_treated = forward_fill_by_group(df_treated, 'MBS_Current_reel_ID')
 
# Step 6: Fill missing values in selected columns using the median of AB_Grade_ID groups
columns_to_fill = [
    'Draw_PS-PD1',
    'Multifractor_1_Long_fibre_fraction',
    'Multifractor_2_long_fibre_fraction',
    'Multifractor_3_long_fibre_fraction']
df_treated = fill_na_with_median(df_treated, columns_to_fill, 'AB_Grade_ID')
 
# =================== FINAL CHECK ===================
print("Final missing value count per column:")
print(df_treated.isna().sum())
 
"""
TURNUP AND CONTINUOUS DATA
"""
df_final_turnup, df_final_continuous = create_turnup_and_continuous_data(df_treated,vars_average= control_vars + response_vars)#,option="median")
#df_final_turnup, df_final_continuous = create_turnup_and_continuous_data(df_treated)#,option="median")
 
# Missing values percentage across splits
quality_missing_percentage_train = df_final_turnup.isnull().mean() * 100
 
# Combine into a single dataframe
df_quality_missing_percentage = pd.DataFrame({
    'Missing Percentage Train': quality_missing_percentage_train
})
 
df_quality_missing_percentage[
    (df_quality_missing_percentage['Missing Percentage Train'] > 0)
]

Step 1: Replace tags with corresponding names in PHD

Step 2: Perform calculations
Step 3: Align dataframe with features from data glossary

Step 4: Clean name of the columns

Number of records initially: (27912, 363)
Number of records after merging quality with process: (27912, 363)
Number of records before removing rows with Grade = 0: (27912, 363)
Number of records after removing rows with Grade = 0: (27912, 363)
Number of records before dropping paperbreaks: (27912, 363)
Number of records after dropping paperbreaks: (18390, 363)
Removed empty columns. Remaining columns: 363
Dropped rows with missing values in ['Starch__€/T_', 'Dry_Strength_Agent__€/T_', 'Fibre_cost__€/T_', 'Combined_cost__€/T_', 'Starch_mass_flow__kg/T_', 'Dry_Strength_Agent_mass_flow__kg/T_']. Remaining rows: 18388


c:\workspace\costimiser\src\process_data_treatment.py:22: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_filled = df_sorted.groupby(group_col).apply(lambda group: group.fillna(method='ffill'))
c:\workspace\costimiser\src\process_data_treatment.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_filled = df_sorted.groupby(group_col).apply(lambda group: group.fillna(method='ffill'))


Forward-filled missing values within each MBS_Current_reel_ID group.
Replaced 999999 and -999999 with NaN column by column.


c:\workspace\costimiser\src\process_data_treatment.py:22: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_filled = df_sorted.groupby(group_col).apply(lambda group: group.fillna(method='ffill'))
c:\workspace\costimiser\src\process_data_treatment.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_filled = df_sorted.groupby(group_col).apply(lambda group: group.fillna(method='ffill'))


Forward-filled missing values within each MBS_Current_reel_ID group.
Median imputation applied. Missing values reduced from 43 to 0.
Final missing value count per column:
Wedge_Time                                     0
MBS_SCT_MD                                  1462
MBS_SCT_CD                                  1462
MBS_Burst                                   1462
MBS_CMT30                                  10780
                                           ...  
Freshwater_retention__l/min_                   0
Freshwater_to_Machine_Tank__l/min_             0
Freshwater_pressure_from_the_city__bar_        0
Waste_steam_flow                               0
Headbox_consistency                            0
Length: 363, dtype: int64
(TURNUP) Number of records after selecting 10min to turnup: (383, 363)

(TURNUP) Number of records after dropping duplicates: (292, 363)

(CONTINUOUS) Number of records after dropping duplicates: (13257, 363)



c:\workspace\costimiser\src\alignment_pipeline.py:85: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_final_turnup_2 = df_final_turnup_2.replace('placeholder', np.nan)


,Missing Percentage Train
MBS_SCT_MD,2.054795
MBS_SCT_CD,2.054795
MBS_Burst,2.054795
MBS_CMT30,55.479452


In [ ]:
add_germany_temperature_column(df_final_turnup, datetime_col="Wedge_Time", inplace = True)

In [ ]:
add_germany_temperature_column(df_final_continuous, datetime_col="Wedge_Time", inplace = True)

In [ ]:
df_final_turnup.to_parquet(f"data/costimier_turnup_{time_ref}.parquet")
df_final_continuous.to_parquet(f"data/costimier_continuous_{time_ref}.parquet")
df_cost_ts.to_parquet(f"data/Costimier_PHD_{time_ref}.parquet")

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

df_to_upload=df_final_turnup
to_parquet=True
try:
    buffer = io.StringIO()
    if to_parquet:
        table = pa.Table.from_pandas(df_to_upload, preserve_index=False)
        sink = pa.BufferOutputStream()
        pq.write_table(table, sink)  # add compression="snappy" if desired
        parquet_bytes = sink.getvalue().to_pybytes()
        buffer = io.BytesIO(parquet_bytes)
    else:
        df_to_upload.to_csv(buffer, index=False)
    s3_client.put_object(Bucket=bucket_name, Key=f"turnup/costimier_turnup_{time_ref}.parquet", Body=buffer.getvalue())
except Exception as e:
    print(f"An error occurred: {e}")

## Current

In [37]:
time_ref="2025-01-01"
df_cost_ts = pd.read_parquet(f"phd/phd_all_casted.parquet",engine="fastparquet")

In [ ]:
import pandas as pd

time_ref="2025-01-01"
df_cost_ts = pd.read_parquet(f"phd/phd.parquet",engine="fastparquet")
#df_cost_ts = pd.read_parquet(f"phd/phd_all_casted.parquet",engine="fastparquet")


df_cost_ts["TimeStamp"]=pd.to_datetime(df_cost_ts["TimeStamp"],format="%d-%b-%Y %H:%M:%S")
df_cost_ts=df_cost_ts.sort_values(by='TimeStamp')
print(f"Time Period: {df_cost_ts['TimeStamp'].min()}, {df_cost_ts['TimeStamp'].max()}")

optvsn_df=pd.read_parquet(f"phd/optvsn_df.parquet")
optvsn_df['MBS_Current_reel_ID']=optvsn_df['MBS_Current_reel_ID'].astype(int)
#optvsn_df['MBS_Current_reel_ID']=optvsn_df['MBS_Current_reel_ID'] - 1
#optvsn_df['MBS_Current_reel_ID']=optvsn_df['MBS_Current_reel_ID'].shift(1)
#optvsn_df['AB_Grade_ID']=optvsn_df['AB_Grade_ID'].shift(1)
optvsn_df['sample_time']=optvsn_df['sample_time'].shift(-1)
optvsn_df=optvsn_df[:-1]

df_cost_ts=df_cost_ts.drop(['QO_SCT_CD','QO_SCT_MD','QO_BERSTWIDERSTAND','QO_CMT'],axis=1)
df_cost_ts=df_cost_ts.merge(optvsn_df[['SCT_CD_value','SCT_MD_value','Burst_value','CMT30_value','MBS_Current_reel_ID']], left_on='QO-REEL-ID-CUR-CALC', right_on='MBS_Current_reel_ID', how="left").rename(columns={"SCT_CD_value":"QO_SCT_CD", "SCT_MD_value":"QO_SCT_MD", "Burst_value":"QO_BERSTWIDERSTAND", "CMT30_value":"QO_CMT"})

Time Period: 2025-01-01 00:00:00, 2026-03-16 17:00:00


In [3]:
df_tags = pd.read_csv("data/tags-v3.csv", sep=';')  # Adjusted to match the CSV format used in the original code
df_tags['TagNumber'] = df_tags['TagNumber'].astype(float)
tag_list = df_tags['Tags'][df_tags['TagNumber'].notna()]
tag_list = tag_list.to_list()

df_gloss = pd.read_csv("data/data_glossary_3.csv", sep=';')

In [4]:
import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np


from src.ingestion_pipeline import ingestion_pipeline_API_alternative
from src.alignment_pipeline import data_alignment, assign_missing_values_to_grades_without_cmt30, drop_grades_equal_to_zero, drop_paperbreaks,create_turnup_and_continuous_data
from src.process_data_treatment  import remove_empty_columns, drop_rows_with_missing_values, forward_fill_by_group, replace_placeholder_values, forward_fill_by_group, fill_na_with_median
from utility import add_germany_temperature_column

In [5]:
def create_turnup_and_continuous_data(df, vars_average=[], option="last"):

    if option=="median":
        df_final_turnup = df.groupby('MBS_Current_reel_ID').median().reset_index()
    else:
        # Select 10min to turnup
        #df_final_turnup = df.groupby('MBS_Current_reel_ID').tail(2).groupby('MBS_Current_reel_ID').head(1)
        df_final_turnup = df.groupby('MBS_Current_reel_ID').tail(10).groupby('MBS_Current_reel_ID').median().reset_index()
    
    for v in vars_average:
        df_final_turnup[v] = df_final_turnup["MBS_Current_reel_ID"].map(df.groupby('MBS_Current_reel_ID', dropna=False)[v].mean())
    
    print(f"(TURNUP) Number of records after selecting 10min to turnup: {df_final_turnup.shape}\n")
    # Drop duplicates
    # df_final_turnup_2 = df_final_turnup.drop_duplicates(subset=['MBS_SCT_MD', 'MBS_SCT_CD', 'MBS_Burst', 'AB_Grade_ID'], keep='first')
    df_final_turnup_2 = df_final_turnup.fillna('placeholder').drop_duplicates(subset=['MBS_SCT_MD', 'MBS_SCT_CD', 'MBS_Burst', 'AB_Grade_ID'], keep='first')
    # Replace the placeholder with NaN
    df_final_turnup_2 = df_final_turnup_2.replace('placeholder', np.nan)
    print(f"(TURNUP) Number of records after dropping duplicates: {df_final_turnup_2.shape}\n")

    # Create dropped duplicates dataframe
    df_dropped_reels = df_final_turnup[~df_final_turnup['MBS_Current_reel_ID'].isin(df_final_turnup_2['MBS_Current_reel_ID'])]
    # Drop duplicates from continuous data
    df_final_continuous = df[df['MBS_Current_reel_ID'].isin(df_final_turnup_2['MBS_Current_reel_ID'])]
    print(f"(CONTINUOUS) Number of records after dropping duplicates: {df_final_continuous.shape}\n")

    return df_final_turnup_2, df_final_continuous

In [6]:
from src.ingestion_pipeline import replace_tags_with_name, combined_calculations, clean_column_names, compare_dataframes

def ingestion_pipeline_API_alternative(df:pd.DataFrame, df_tags, df_gloss) -> pd.DataFrame:
    df["R2NAE20CF905"]=df["R2NAE20CF905"].fillna(0)
    df_initial = df.copy()
    print("Step 1: Replace tags with corresponding names in PHD \n")
    df = replace_tags_with_name(df, df_tags)
    df = df.copy()
    print("Step 2: Perform calculations")
    df = combined_calculations(df)
    #print("Step 3: Align dataframe with features from data glossary \n")
    #df = alligning_dataframe(df, df_gloss)
    print("Step 4: Clean name of the columns \n")
    df = clean_column_names(df)
    print("Before and after of ingestion pipeline: \n")
    compare_dataframes(df_initial, df, "Ingestion Pipeline")
    return df

In [9]:
df_process_ts  = ingestion_pipeline_API_alternative(df_cost_ts, df_tags, df_gloss)
df_process_ts['Wedge_Time'] = pd.to_datetime(df_process_ts['Wedge_Time'])

quality_list = ['MBS_SCT_MD', 'MBS_SCT_CD', 'MBS_Burst', 'MBS_CMT30']
if time_ref=="2025-01-01":
    data_aligned_continuous = df_process_ts
else:
    data_aligned_continuous = data_alignment(df_process_ts, quality_list)
#data_aligned_continuous = df_process_ts

""" 
ASSIGN MISSING VALUES TO QUALITY VALUES FOR GRADES THAT DON'T HAVE CMT30
"""
# Defining grades that dont have CMT30
grade_list = [3600075, 3600085,6010075,6010080,6010085,6010090,6010095,3200100,3200115,3200125,3300085,
            3300090,3300095,3300100,3300105,3300110,3300115,3300120,3300125,3300130,3300135]
data_aligned_continuous = assign_missing_values_to_grades_without_cmt30(data_aligned_continuous, grade_list)

# Drop zero grades
data_aligned_continuous = drop_grades_equal_to_zero(data_aligned_continuous)

# Drop rows with paperbreaks
data_aligned_continuous = drop_paperbreaks(data_aligned_continuous)

# Step 1: Remove columns that are completely missing
df_treated = remove_empty_columns(data_aligned_continuous)

# Step 2: Drop rows where specific columns have NaN values
columns_to_verify_missing = [
    "Starch__€/T_",
    "Dry_Strength_Agent__€/T_",
    "Fibre_cost__€/T_",
    "Combined_cost__€/T_",
    "Starch_mass_flow__kg/T_",
    "Dry_Strength_Agent_mass_flow__kg/T_"]
df_treated = drop_rows_with_missing_values(df_treated, columns_to_verify_missing)

# Step 3: Forward-fill within each MBS_Current_reel_ID group
df_treated = forward_fill_by_group(df_treated, 'MBS_Current_reel_ID')

# Step 4: Replace placeholder values (999999) with NaN
df_treated = replace_placeholder_values(df_treated)

# Step 5: Forward-fill again within each MBS_Current_reel_ID group (for NaNs introduced by replacing 999999)
df_treated = forward_fill_by_group(df_treated, 'MBS_Current_reel_ID')

# Step 6: Fill missing values in selected columns using the median of AB_Grade_ID groups
columns_to_fill = [
    'Draw_PS-PD1',
    'Multifractor_1_Long_fibre_fraction',
    'Multifractor_2_long_fibre_fraction',
    'Multifractor_3_long_fibre_fraction']
df_treated = fill_na_with_median(df_treated, columns_to_fill, 'AB_Grade_ID')

# =================== FINAL CHECK ===================
print("Final missing value count per column:")
print(df_treated.isna().sum())

"""
TURNUP AND CONTINUOUS DATA
"""
df_final_turnup, df_final_continuous = create_turnup_and_continuous_data(df_treated)#,option="median")
#df_final_turnup, df_final_continuous = create_turnup_and_continuous_data(df_treated)#,option="median")

# Missing values percentage across splits
quality_missing_percentage_train = df_final_turnup.isnull().mean() * 100

# Combine into a single dataframe
df_quality_missing_percentage = pd.DataFrame({
    'Missing Percentage Train': quality_missing_percentage_train
})

df_quality_missing_percentage[
    (df_quality_missing_percentage['Missing Percentage Train'] > 0)
]

Step 1: Replace tags with corresponding names in PHD 

Step 2: Perform calculations
Step 4: Clean name of the columns 

Before and after of ingestion pipeline: 

                        Name          Shape  Missing Values (%)  \
0  Before Ingestion Pipeline  (645371, 399)            1.581969   
1   After Ingestion Pipeline  (645371, 470)            1.629205   

   Infinite Values (%)  Substituted infinite values  Duplicate Rows  
0                  0.0                     0.000000            1576  
1                  0.0                     0.292826            1576  
Number of records before removing rows with Grade = 0: (645371, 470)
Number of records after removing rows with Grade = 0: (645371, 470)
Number of records before dropping paperbreaks: (645371, 470)
Number of records after dropping paperbreaks: (522389, 470)
Removed empty columns. Remaining columns: 466
Dropped rows with missing values in ['Starch__€/T_', 'Dry_Strength_Agent__€/T_', 'Fibre_cost__€/T_', 'Combined_cost__€/T_'

c:\workspace\costimiser\src\process_data_treatment.py:22: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_filled = df_sorted.groupby(group_col).apply(lambda group: group.fillna(method='ffill'))
c:\workspace\costimiser\src\process_data_treatment.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_filled = df_sorted.groupby(group_col).apply(lambda group: group.fillna(method='ffill'))


Forward-filled missing values within each MBS_Current_reel_ID group.
Replaced 999999 and -999999 with NaN column by column.


c:\workspace\costimiser\src\process_data_treatment.py:22: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_filled = df_sorted.groupby(group_col).apply(lambda group: group.fillna(method='ffill'))
c:\workspace\costimiser\src\process_data_treatment.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_filled = df_sorted.groupby(group_col).apply(lambda group: group.fillna(method='ffill'))


Forward-filled missing values within each MBS_Current_reel_ID group.
Median imputation applied. Missing values reduced from 157 to 0.
Final missing value count per column:
Wedge_Time                     0
MBS_Current_reel_ID            0
AB_Grade_ID                    0
MBS_SCT_CD                134561
MBS_SCT_MD                134610
                           ...  
Electricity__€/T_              0
Steam__€/T_                    0
Fibre_cost__€/T_               0
Combined_cost__€/T_            0
Total_SQM_cost__€/hm2_         0
Length: 466, dtype: int64
(TURNUP) Number of records after selecting 10min to turnup: (10682, 466)

(TURNUP) Number of records after dropping duplicates: (7256, 466)



C:\Users\gbmiggon\AppData\Local\Temp\ipykernel_20208\3346900853.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_final_turnup_2 = df_final_turnup_2.replace('placeholder', np.nan)


(CONTINUOUS) Number of records after dropping duplicates: (385764, 466)



,Missing Percentage Train
MBS_SCT_CD,0.234289
MBS_SCT_MD,0.248071
MBS_Burst,0.261852
MBS_CMT30,86.645535


In [21]:
add_germany_temperature_column(df_final_turnup, datetime_col="Wedge_Time", inplace = True)

c:\workspace\costimiser\utility.py:4848: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[out_col] = ts.map(lookup)


In [ ]:
df_final_turnup.to_parquet(f"data/costimier_turnup.parquet")
df_final_continuous.to_parquet(f"data/costimier_continuous.parquet")
df_cost_ts.to_parquet(f"data/Costimier_PHD.parquet")

In [26]:
import pyarrow as pa
import pyarrow.parquet as pq

df_to_upload=df_final_turnup
to_parquet=True
try:
    buffer = io.StringIO()
    if to_parquet:
        table = pa.Table.from_pandas(df_to_upload, preserve_index=False)
        sink = pa.BufferOutputStream()
        pq.write_table(table, sink)  # add compression="snappy" if desired
        parquet_bytes = sink.getvalue().to_pybytes()
        buffer = io.BytesIO(parquet_bytes)
    else:
        df_to_upload.to_csv(buffer, index=False)
    s3_client.put_object(Bucket=bucket_name, Key=f"turnup/costimier_turnup_.parquet", Body=buffer.getvalue())
except Exception as e:
    print(f"An error occurred: {e}")

## All tags

In [ ]:
optvsn_df=pd.read_parquet(f"phd/optvsn_df.parquet")
optvsn_df['MBS_Current_reel_ID']=optvsn_df['MBS_Current_reel_ID'].astype(int)
optvsn_df['sample_time']=optvsn_df['sample_time'].shift(-1)
optvsn_df=optvsn_df[:-1]

In [55]:
import pandas as pd

time_ref="20251224"
df_cost_ts = pd.read_parquet(f"data-v3/raw-fetched-data/phd_data_{time_ref}.parquet",engine="fastparquet")
df_cost_ts['QO-REEL-ID-CUR-CALC']=df_cost_ts['QO-REEL-ID-CUR-CALC'].astype(int)


df_cost_ts["TimeStamp"]=pd.to_datetime(df_cost_ts["TimeStamp"],format="%d-%b-%Y %H:%M:%S")
df_cost_ts=df_cost_ts.sort_values(by='TimeStamp')
print(f"Time Period: {df_cost_ts['TimeStamp'].min()}, {df_cost_ts['TimeStamp'].max()}")

df_cost_ts=df_cost_ts.drop(['QO_SCT_CD','QO_SCT_MD','QO_BERSTWIDERSTAND','QO_CMT'],axis=1)
df_cost_ts=df_cost_ts.merge(optvsn_df[['SCT_CD_value','SCT_MD_value','Burst_value','CMT30_value','MBS_Current_reel_ID']], left_on='QO-REEL-ID-CUR-CALC', right_on='MBS_Current_reel_ID', how="left").rename(columns={"SCT_CD_value":"QO_SCT_CD", "SCT_MD_value":"QO_SCT_MD", "Burst_value":"QO_BERSTWIDERSTAND", "CMT30_value":"QO_CMT"})

Time Period: 2025-12-24 00:00:00, 2025-12-25 00:00:00


In [57]:
df_process_ts  = ingestion_pipeline_API_alternative(df_cost_ts, df_tags, df_gloss)

Step 1: Replace tags with corresponding names in PHD 

Step 2: Perform calculations
Step 4: Clean name of the columns 

Before and after of ingestion pipeline: 

                        Name         Shape  Missing Values (%)  \
0  Before Ingestion Pipeline  (1441, 2293)            7.721597   
1   After Ingestion Pipeline   (1441, 439)            5.156189   

   Infinite Values (%)  Substituted infinite values  Duplicate Rows  
0                  0.0                     0.000000               0  
1                  0.0                     3.775441               0  


In [61]:
from pathlib import Path
import re
import pandas as pd
 
 
input_dir = Path("data-v3/raw-fetched-data")
output_base = Path("data-v3/aggregated")
 
pattern = re.compile(r"phd_data_(\d{8})\.parquet$")
 
 
for file_path in sorted(input_dir.glob("phd_data_*.parquet")):
    m = pattern.search(file_path.name)
    if not m:
        print(f"Skipping unexpected filename: {file_path.name}")
        continue
 
    time_ref = m.group(1)   # e.g. 20251231
    year = time_ref[:4]
    month = time_ref[4:6]
    day = time_ref[6:8]
 
    print(f"\nProcessing {file_path.name} ...")
 
    # Read raw file
    df_cost_ts = pd.read_parquet(file_path, engine="fastparquet")
 
    # Basic cleaning
    df_cost_ts['QO-REEL-ID-CUR-CALC'] = df_cost_ts['QO-REEL-ID-CUR-CALC'].astype(int)
    df_cost_ts["TimeStamp"] = pd.to_datetime(
        df_cost_ts["TimeStamp"],
        format="%d-%b-%Y %H:%M:%S"
    )
    df_cost_ts = df_cost_ts.sort_values(by="TimeStamp")
 
    print(f"Time Period: {df_cost_ts['TimeStamp'].min()}, {df_cost_ts['TimeStamp'].max()}")
 
    # Drop old lab columns if present
    cols_to_drop = ['QO_SCT_CD', 'QO_SCT_MD', 'QO_BERSTWIDERSTAND', 'QO_CMT']
    df_cost_ts = df_cost_ts.drop(columns=[c for c in cols_to_drop if c in df_cost_ts.columns])
 
    # Merge Optivision values
    df_cost_ts = (
        df_cost_ts.merge(
            optvsn_df[['SCT_CD_value', 'SCT_MD_value', 'Burst_value', 'CMT30_value', 'MBS_Current_reel_ID']],
            left_on='QO-REEL-ID-CUR-CALC',
            right_on='MBS_Current_reel_ID',
            how="left"
        )
        .rename(columns={
            "SCT_CD_value": "QO_SCT_CD",
            "SCT_MD_value": "QO_SCT_MD",
            "Burst_value": "QO_BERSTWIDERSTAND",
            "CMT30_value": "QO_CMT"
        })
    )
 
    # Ingestion pipeline
    df_process_ts = ingestion_pipeline_API_alternative(df_cost_ts, df_tags, df_gloss)
    df_process_ts['Wedge_Time'] = pd.to_datetime(df_process_ts['Wedge_Time'])
 
    data_aligned_continuous = df_process_ts
 
    grade_list = [
        3600075, 3600085, 6010075, 6010080, 6010085, 6010090, 6010095,
        3200100, 3200115, 3200125, 3300085, 3300090, 3300095, 3300100,
        3300105, 3300110, 3300115, 3300120, 3300125, 3300130, 3300135
    ]
 
    data_aligned_continuous = assign_missing_values_to_grades_without_cmt30(
        data_aligned_continuous,
        grade_list
    )
 
    # Output path
    output_dir = output_base / year / month / day
    output_dir.mkdir(parents=True, exist_ok=True)
 
    output_file = output_dir / f"aggregated_{time_ref}.parquet"
    data_aligned_continuous.to_parquet(output_file, index=False)
 
    print(f"Saved: {output_file}")


Processing phd_data_20250401.parquet ...
Time Period: 2025-04-01 00:00:00, 2025-04-02 00:00:00
Step 1: Replace tags with corresponding names in PHD 

Step 2: Perform calculations
Step 4: Clean name of the columns 

Before and after of ingestion pipeline: 

                        Name         Shape  Missing Values (%)  \
0  Before Ingestion Pipeline  (1441, 2293)            7.719175   
1   After Ingestion Pipeline   (1441, 439)            0.740437   

   Infinite Values (%)  Substituted infinite values  Duplicate Rows  
0                  0.0                          0.0               0  
1                  0.0                          0.0               0  
Saved: data-v3\aggregated\2025\04\01\aggregated_20250401.parquet

Processing phd_data_20250402.parquet ...
Time Period: 2025-04-02 00:00:00, 2025-04-03 00:00:00
Step 1: Replace tags with corresponding names in PHD 

Step 2: Perform calculations
Step 4: Clean name of the columns 

Before and after of ingestion pipeline: 

          